# Day 13: Computer Vision with OpenCV and CNNs  
## Image Processing, Kernels, Edges, Contours, Cropping, and CNN Architecture

This notebook is designed to make students feel excited about images and computer vision.

The goal is not to overload them with math. The goal is to help them **see** what happens to an image step by step.

Students will learn:

- How images are stored as arrays
- How to load an image in Colab/Jupyter
- Why OpenCV uses BGR instead of RGB
- How grayscale conversion works
- How blur removes noise
- How kernels create effects like edge detection and sharpening
- How thresholding separates foreground from background
- How contours detect objects
- How bounding boxes crop regions of interest
- How a CNN architecture is built using Keras

Important Colab note:

`cv2.imshow()` can crash Colab because it tries to open a GUI window on Google's server.

So we use Matplotlib:

```python
plt.imshow(...)
plt.show()
```

---

# 0. Setup

If OpenCV is not installed, run:

```python
!pip install opencv-python
```

In Google Colab, OpenCV is usually already available.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

print("Day 13 Computer Vision Environment Initialized.")
print("OpenCV version:", cv2.__version__)

Day 13 Computer Vision Environment Initialized.
OpenCV version: 4.11.0


---

# 1. Load an Image: Upload or Use an Image URL

You said you may take an image from Google.

There are two easy ways to use images in Colab:

## Option A: Upload from your computer

Students can upload any image file.

## Option B: Use an image URL

Students can paste a direct image URL.

For teaching, uploading is usually safer because many Google image URLs are not direct image links.

Run only one method.

## Option A: Upload Image in Google Colab

Run this cell in Colab and upload an image.

Supported formats: `.jpg`, `.jpeg`, `.png`

In [2]:
# OPTION A: Upload image in Google Colab

from google.colab import files
import io
from PIL import Image

uploaded = files.upload()

file_name = list(uploaded.keys())[0]
pil_image = Image.open(io.BytesIO(uploaded[file_name])).convert("RGB")

image_rgb = np.array(pil_image)
image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

plt.figure(figsize=(7, 5))
plt.imshow(image_rgb)
plt.title("Uploaded Image")
plt.axis("off")
plt.show()

print("Image shape:", image_rgb.shape)

ModuleNotFoundError: No module named 'google.colab'

## Option B: Load Image from a Direct URL

Use this only if you have a direct image URL ending with `.jpg`, `.jpeg`, or `.png`.

If this cell fails, use Option A upload instead.

In [ ]:
# OPTION B: Load image from a direct URL

# Uncomment and replace this URL with a direct image URL.
# image_url = "https://example.com/image.jpg"

# import requests
# from PIL import Image
# import io

# response = requests.get(image_url)
# pil_image = Image.open(io.BytesIO(response.content)).convert("RGB")

# image_rgb = np.array(pil_image)
# image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

# plt.figure(figsize=(7, 5))
# plt.imshow(image_rgb)
# plt.title("Image Loaded from URL")
# plt.axis("off")
# plt.show()

# print("Image shape:", image_rgb.shape)

## Backup Demo Image

If upload or URL does not work, this cell creates a colorful synthetic image so the rest of the notebook still runs.

Only run this if you do not have `image_rgb` already.

In [ ]:
# Backup synthetic image for classroom demo

try:
    image_rgb
    print("Using uploaded/URL image.")
except NameError:
    print("No image found. Creating synthetic demo image.")
    
    image_rgb = np.zeros((300, 400, 3), dtype=np.uint8)
    image_rgb[:] = [230, 230, 230]
    
    cv2.rectangle(image_rgb, (40, 50), (160, 180), (255, 80, 80), -1)
    cv2.circle(image_rgb, (290, 120), 60, (80, 180, 255), -1)
    cv2.putText(image_rgb, "AI", (145, 260), cv2.FONT_HERSHEY_SIMPLEX, 3, (40, 40, 40), 8)
    
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    
    plt.figure(figsize=(7, 5))
    plt.imshow(image_rgb)
    plt.title("Synthetic Demo Image")
    plt.axis("off")
    plt.show()

print("Image shape:", image_rgb.shape)

---

# 2. OpenCV BGR vs Matplotlib RGB

This is one of the most important OpenCV beginner mistakes.

OpenCV reads images in **BGR** order: Blue, Green, Red.

Matplotlib expects **RGB** order: Red, Green, Blue.

If students display an OpenCV image directly in Matplotlib without conversion, the colors look wrong.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.imshow(image_bgr)
ax1.set_title("Wrong Display: BGR shown as RGB")
ax1.axis("off")

ax2.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
ax2.set_title("Correct Display: BGR converted to RGB")
ax2.axis("off")

plt.show()

---

# 3. Images Are NumPy Arrays

An image is a NumPy array.

For a color image:

```text
height × width × 3
```

The 3 channels are Red, Green, and Blue.

Each pixel value usually ranges from 0 to 255.

In [ ]:
print("Image type:", type(image_rgb))
print("Image shape:", image_rgb.shape)
print("Minimum pixel value:", image_rgb.min())
print("Maximum pixel value:", image_rgb.max())

print("\nTop-left pixel RGB value:")
print(image_rgb[0, 0])

## Visualizing Color Channels Separately

This cell separates the image into Red, Green, and Blue channels.

Students can clearly see how each channel contributes to the final image.

In [ ]:
red_channel = image_rgb[:, :, 0]
green_channel = image_rgb[:, :, 1]
blue_channel = image_rgb[:, :, 2]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(red_channel, cmap="Reds")
axes[0].set_title("Red Channel")
axes[0].axis("off")

axes[1].imshow(green_channel, cmap="Greens")
axes[1].set_title("Green Channel")
axes[1].axis("off")

axes[2].imshow(blue_channel, cmap="Blues")
axes[2].set_title("Blue Channel")
axes[2].axis("off")

plt.show()

---

# 4. Grayscale Conversion

Many traditional computer vision algorithms work better on grayscale images.

A grayscale image has only one channel.

This makes processing simpler.

In [ ]:
gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.imshow(image_rgb)
ax1.set_title("Original Color Image")
ax1.axis("off")

ax2.imshow(gray, cmap="gray")
ax2.set_title("Grayscale Image")
ax2.axis("off")

plt.show()

print("Color image shape:", image_rgb.shape)
print("Grayscale image shape:", gray.shape)

---

# 5. The Math of Convolution: Manual Filter

A convolution filter is a small matrix called a kernel.

The kernel slides over the image and transforms it.

Different kernels create different effects:

- Blur
- Sharpen
- Edge detection
- Emboss
- Outline

This cell uses a simple edge detection kernel on a synthetic square image so students can clearly see what the filter does.

In [ ]:
from scipy.signal import convolve2d

synthetic_image = np.zeros((10, 10))
synthetic_image[3:8, 3:8] = 255

edge_kernel = np.array([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1]
])

feature_map = convolve2d(synthetic_image, edge_kernel, mode="valid")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))

ax1.imshow(synthetic_image, cmap="gray")
ax1.set_title("Original 2D Array")
ax1.axis("off")

ax2.imshow(feature_map, cmap="gray")
ax2.set_title("After Convolution: Edges Found")
ax2.axis("off")

plt.show()

## Kernel Playground

This is a fun classroom cell.

Students can see how different kernels change the same image.

We apply blur, sharpen, edge detection, and emboss filters.

In [ ]:
gray_small = cv2.resize(gray, (250, 250))

kernels = {
    "Blur": np.ones((3, 3)) / 9,
    "Sharpen": np.array([[0, -1, 0],
                         [-1, 5, -1],
                         [0, -1, 0]]),
    "Edge Detection": np.array([[-1, -1, -1],
                                [-1, 8, -1],
                                [-1, -1, -1]]),
    "Emboss": np.array([[-2, -1, 0],
                        [-1, 1, 1],
                        [0, 1, 2]])
}

fig, axes = plt.subplots(1, len(kernels) + 1, figsize=(16, 4))

axes[0].imshow(gray_small, cmap="gray")
axes[0].set_title("Original")
axes[0].axis("off")

for ax, (name, kernel) in zip(axes[1:], kernels.items()):
    filtered = convolve2d(gray_small, kernel, mode="same", boundary="symm")
    ax.imshow(filtered, cmap="gray")
    ax.set_title(name)
    ax.axis("off")

plt.show()

---

# 6. Blur: Removing Noise

Images often contain noise from low camera quality, poor lighting, scanning artifacts, compression, or motion blur.

Gaussian Blur smooths the image and reduces small noise.

In [ ]:
blurred = cv2.GaussianBlur(gray, (7, 7), 0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.imshow(gray, cmap="gray")
ax1.set_title("Before Blur")
ax1.axis("off")

ax2.imshow(blurred, cmap="gray")
ax2.set_title("After Gaussian Blur")
ax2.axis("off")

plt.show()

---

# 7. Edge Detection with Canny

Canny edge detection finds strong boundaries in an image.

It is useful for object outlines, document detection, lane detection, shape detection, and preprocessing before contour detection.

In [ ]:
edges = cv2.Canny(blurred, threshold1=50, threshold2=150)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.imshow(image_rgb)
ax1.set_title("Original Image")
ax1.axis("off")

ax2.imshow(edges, cmap="gray")
ax2.set_title("Canny Edge Detection")
ax2.axis("off")

plt.show()

## Edge Threshold Comparison

This cell shows how changing Canny thresholds changes the edge output.

In [ ]:
threshold_pairs = [(30, 100), (50, 150), (100, 200)]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (t1, t2) in zip(axes, threshold_pairs):
    edge_result = cv2.Canny(blurred, t1, t2)
    ax.imshow(edge_result, cmap="gray")
    ax.set_title(f"Canny: {t1}, {t2}")
    ax.axis("off")

plt.show()

---

# 8. Synthesizing a Scanned Document

Since we are in a cloud environment, we can generate a synthetic noisy scanned assignment.

The image contains:

- Noisy paper background
- Dark answer box
- Dark answer circle
- Grade text

In [ ]:
np.random.seed(42)

scanned_doc = np.random.randint(180, 220, (300, 400), dtype=np.uint8)

cv2.rectangle(scanned_doc, (50, 50), (150, 100), 20, -1)
cv2.circle(scanned_doc, (300, 150), 40, 10, -1)
cv2.putText(scanned_doc, "A+", (100, 250), cv2.FONT_HERSHEY_SIMPLEX, 3, 0, 8)

noise = np.random.randint(-30, 30, (300, 400), dtype=np.int8)
scanned_doc = np.clip(scanned_doc.astype(np.int16) + noise, 0, 255).astype(np.uint8)

plt.figure(figsize=(7, 5))
plt.imshow(scanned_doc, cmap="gray", vmin=0, vmax=255)
plt.title("Raw Scanned Assignment with Noise")
plt.axis("off")
plt.show()

---

# 9. OpenCV Preprocessing: Blur and Thresholding

Before feeding a scanned document to a CNN or OCR model, we often clean it.

Gaussian blur reduces noise.

Thresholding separates ink from paper.

We use inverse thresholding so ink becomes white and paper becomes black.

In [ ]:
blurred_doc = cv2.GaussianBlur(scanned_doc, (5, 5), 0)

_, thresholded = cv2.threshold(
    blurred_doc,
    120,
    255,
    cv2.THRESH_BINARY_INV
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

ax1.imshow(blurred_doc, cmap="gray")
ax1.set_title("After Gaussian Blur")
ax1.axis("off")

ax2.imshow(thresholded, cmap="gray")
ax2.set_title("After Inverse Thresholding")
ax2.axis("off")

plt.show()

## Threshold Value Comparison

This helps students understand that threshold value matters.

In [ ]:
threshold_values = [80, 120, 160]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, value in zip(axes, threshold_values):
    _, temp_thresh = cv2.threshold(blurred_doc, value, 255, cv2.THRESH_BINARY_INV)
    ax.imshow(temp_thresh, cmap="gray")
    ax.set_title(f"Threshold = {value}")
    ax.axis("off")

plt.show()

---

# 10. Finding Contours and Bounding Boxes

Contours are boundaries around connected objects.

Once we threshold the image, OpenCV can detect separate objects.

We can then draw bounding boxes around them.

This is useful for detecting answer regions, cropping handwritten answers, finding objects, and preparing image patches for CNNs.

In [ ]:
contours, hierarchy = cv2.findContours(
    thresholded,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

output_img = cv2.cvtColor(scanned_doc, cv2.COLOR_GRAY2BGR)

valid_contours = []

for i, contour in enumerate(contours):
    if cv2.contourArea(contour) > 500:
        x, y, w, h = cv2.boundingRect(contour)
        valid_contours.append((x, y, w, h))
        
        cv2.rectangle(output_img, (x, y), (x+w, y+h), (0, 255, 0), 3)
        cv2.putText(output_img, f"Object {len(valid_contours)}", (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

plt.figure(figsize=(8, 5))
plt.imshow(cv2.cvtColor(output_img, cv2.COLOR_BGR2RGB))
plt.title(f"OpenCV Pipeline Complete: Found {len(valid_contours)} main objects")
plt.axis("off")
plt.show()

print("Bounding boxes:", valid_contours)

## Cropping Detected Objects

This cell crops each detected object.

In a real system, each crop could be resized and passed into a CNN.

In [ ]:
if len(valid_contours) > 0:
    fig, axes = plt.subplots(1, len(valid_contours), figsize=(4 * len(valid_contours), 4))
    
    if len(valid_contours) == 1:
        axes = [axes]
    
    for ax, (x, y, w, h) in zip(axes, valid_contours):
        crop = thresholded[y:y+h, x:x+w]
        ax.imshow(crop, cmap="gray")
        ax.set_title(f"Crop {x},{y},{w},{h}")
        ax.axis("off")
    
    plt.show()
else:
    print("No large contours found.")

---

# 11. Morphological Operations

Morphological operations clean binary images.

Useful operations:

| Operation | Use |
|---|---|
| Erosion | Shrinks white regions |
| Dilation | Expands white regions |
| Opening | Removes small noise |
| Closing | Fills small holes |

In [ ]:
morph_kernel = np.ones((5, 5), np.uint8)

eroded = cv2.erode(thresholded, morph_kernel, iterations=1)
dilated = cv2.dilate(thresholded, morph_kernel, iterations=1)
opened = cv2.morphologyEx(thresholded, cv2.MORPH_OPEN, morph_kernel)
closed = cv2.morphologyEx(thresholded, cv2.MORPH_CLOSE, morph_kernel)

images = [thresholded, eroded, dilated, opened, closed]
titles = ["Original Threshold", "Erosion", "Dilation", "Opening", "Closing"]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")

plt.show()

---

# 12. Building a CNN Architecture in Keras

Now we build a CNN architecture.

A CNN has two major parts.

## Part 1: Feature Extraction

This part uses `Conv2D` and `MaxPooling2D`.

It learns edges, textures, shapes, and patterns.

## Part 2: Classification

This part uses `Flatten` and `Dense`.

It converts extracted features into a prediction.

This model is designed for 64×64 grayscale images.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

cnn_grader = Sequential([
    Conv2D(32, (3, 3), activation="relu", input_shape=(64, 64, 1)),
    MaxPooling2D(pool_size=(2, 2)),
    
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D(pool_size=(2, 2)),
    
    Flatten(),
    
    Dense(128, activation="relu"),
    Dense(1, activation="sigmoid")
])

cnn_grader.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

cnn_grader.summary()

## CNN Shape Explanation

For a beginner, explain it like this:

```text
Image enters → Convolution finds features → Pooling shrinks image → Flatten converts to list → Dense layer classifies
```

A CNN does not manually require us to write edge filters.

It learns useful filters automatically during training.

---

# 13. Preparing One Crop for CNN Input

CNNs expect images in a fixed shape.

Our model expects:

```text
64 × 64 × 1
```

So every cropped object should be resized, normalized, and reshaped.

In [ ]:
if len(valid_contours) > 0:
    x, y, w, h = valid_contours[0]
    crop = thresholded[y:y+h, x:x+w]
    
    resized_crop = cv2.resize(crop, (64, 64))
    normalized_crop = resized_crop / 255.0
    
    cnn_input = normalized_crop.reshape(1, 64, 64, 1)
    
    plt.figure(figsize=(4, 4))
    plt.imshow(resized_crop, cmap="gray")
    plt.title("CNN-Ready 64x64 Crop")
    plt.axis("off")
    plt.show()
    
    print("CNN input shape:", cnn_input.shape)
else:
    print("No crop available. Run contour detection first.")

---

# Day 13 Hands-On Coding Test

The following problems are for students to solve independently.

No solutions are provided in this test section.

# Test 1: Easy  
## Grayscale and Edge Detection

Using the variable `image_bgr` from this notebook:

1. Convert the image to grayscale
2. Apply Gaussian blur with kernel size `(5, 5)`
3. Apply Canny edge detection with thresholds `50` and `150`
4. Display the grayscale image and edge image side by side

In [ ]:
# Test 1 Student Code

import cv2
import matplotlib.pyplot as plt

# Use image_bgr from the notebook

# Write your solution here

---

# Test 2: Medium  
## Thresholding and Contours

Using the synthetic `scanned_doc` image:

1. Apply Gaussian blur
2. Apply inverse thresholding
3. Find external contours
4. Draw bounding boxes only for contours with area greater than `500`
5. Display the final image with bounding boxes

In [ ]:
# Test 2 Student Code

import cv2
import matplotlib.pyplot as plt

# Use scanned_doc from the notebook

# Write your solution here

---

# Test 3: Hard  
## Build a CNN Architecture

Build a CNN for 128×128 grayscale image classification.

The architecture should be:

1. Conv2D with 32 filters, kernel size `(3, 3)`, ReLU, input shape `(128, 128, 1)`
2. MaxPooling2D with pool size `(2, 2)`
3. Conv2D with 64 filters, kernel size `(3, 3)`, ReLU
4. MaxPooling2D with pool size `(2, 2)`
5. Conv2D with 128 filters, kernel size `(3, 3)`, ReLU
6. MaxPooling2D with pool size `(2, 2)`
7. Flatten
8. Dense layer with 128 neurons and ReLU
9. Dense output layer with 1 neuron and sigmoid
10. Compile using Adam optimizer and binary cross-entropy loss
11. Print the model summary

In [ ]:
# Test 3 Student Code

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

# Write your solution here

---

# End of Day 13 Notebook

By the end of this notebook, students should understand:

- How images are represented as NumPy arrays
- Why OpenCV uses BGR and Matplotlib uses RGB
- How to safely display images in Colab
- How grayscale conversion works
- How kernels transform images
- How blur reduces noise
- How Canny detects edges
- How thresholding separates foreground from background
- How contours locate objects
- How bounding boxes and crops are created
- How crops can be prepared for CNN input
- How CNN architectures are built with Keras

This notebook connects OpenCV preprocessing with CNN-based deep learning.